# Bac 3AS Filter Validation & Analysis

This notebook provides comprehensive analysis of the balanced Bac filter results.

**Sections:**
1. Load and explore filtered data
2. Overall statistics
3. Manual review metrics (if available)
4. Visualizations
5. Error analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

## 1. Load Filtered Data

In [ ]:
# Load main filtered dataset
filtered_path = DATA_DIR / 'processed' / 'videos_bac_balanced.csv'

if filtered_path.exists():
    df = pd.read_csv(filtered_path)
    print(f"✓ Loaded {len(df):,} videos from {filtered_path.name}")
    print(f"\nColumns: {list(df.columns)}")
else:
    print(f"❌ File not found: {filtered_path}")
    print("Run: uv run python run_pipeline.py filter_data")

In [ ]:
# Load validation sample (if exists)
validation_path = DATA_DIR / 'validation' / 'sample_for_manual_review.csv'

if validation_path.exists():
    val_df = pd.read_csv(validation_path)
    print(f"✓ Loaded {len(val_df):,} validation samples")
    
    # Check if manual review has been done
    manual_reviewed = val_df['manual_is_bac'].notna().sum()
    if manual_reviewed > 0:
        print(f"✓ {manual_reviewed} videos have been manually reviewed")
    else:
        print("⚠️  No manual reviews yet. Fill 'manual_is_bac' column for accuracy metrics.")
else:
    val_df = None
    print("⚠️  No validation samples found")
    print("Run: uv run python run_pipeline.py filter_data --validate")

In [ ]:
# Load channel priors
priors_path = DATA_DIR / 'processed' / 'channel_priors.csv'

if priors_path.exists():
    priors_df = pd.read_csv(priors_path)
    print(f"✓ Loaded channel priors for {len(priors_df)} channels")
else:
    priors_df = None
    print("⚠️  Channel priors not found")

## 2. Overall Statistics

In [ ]:
print("="*60)
print("FILTER RESULTS SUMMARY")
print("="*60)

total = len(df)
bac_count = df['is_bac_3as'].sum()
bac_pct = (bac_count / total) * 100

print(f"\nTotal videos: {total:,}")
print(f"Bac 3AS videos: {bac_count:,} ({bac_pct:.1f}%)")
print(f"Rejected videos: {total - bac_count:,} ({100-bac_pct:.1f}%)")

print("\n" + "-"*60)
print("BY CATEGORY")
print("-"*60)
print(df['filter_category'].value_counts().to_string())

print("\n" + "-"*60)
print("CONFIDENCE STATISTICS")
print("-"*60)
print(df.groupby('filter_category')['filter_confidence'].describe()[['mean', 'std', 'min', 'max']])

In [ ]:
# Subject distribution (Bac videos only)
print("\n" + "="*60)
print("SUBJECT DISTRIBUTION (BAC 3AS VIDEOS)")
print("="*60)
bac_videos = df[df['is_bac_3as'] == True]
print(bac_videos['subject'].value_counts().to_string())

In [ ]:
# Filter reason breakdown
print("\n" + "="*60)
print("FILTER DECISION REASONS (TOP 10)")
print("="*60)
print(df['filter_reason'].value_counts().head(10).to_string())

## 3. Manual Review Metrics (if available)

In [ ]:
if val_df is not None and val_df['manual_is_bac'].notna().sum() > 0:
    print("="*60)
    print("MANUAL REVIEW ACCURACY METRICS")
    print("="*60)
    
    reviewed = val_df.dropna(subset=['manual_is_bac'])
    
    # Calculate metrics by category
    metrics = {}
    for category in ['bac_3as', 'bac_3as_ambiguous', 'non_bac', 'unknown']:
        cat_df = reviewed[reviewed['sample_type'] == category]
        
        if len(cat_df) > 0:
            if 'bac' in category:
                # Precision: what % of predicted Bac are actually Bac?
                precision = cat_df['manual_is_bac'].sum() / len(cat_df)
                metrics[category] = {'metric': 'Precision', 'value': precision, 'n': len(cat_df)}
                print(f"\n{category}:")
                print(f"  Precision: {precision:.1%} (n={len(cat_df)})")
            else:
                # False Negative Rate: what % of rejected are actually Bac?
                fnr = cat_df['manual_is_bac'].sum() / len(cat_df)
                metrics[category] = {'metric': 'FNR', 'value': fnr, 'n': len(cat_df)}
                print(f"\n{category}:")
                print(f"  False Negative Rate: {fnr:.1%} (n={len(cat_df)})")
    
    # Overall accuracy
    # For is_bac_3as=True samples, correct = manual_is_bac=True
    # For is_bac_3as=False samples, correct = manual_is_bac=False
    reviewed['correct'] = reviewed['is_bac_3as'] == reviewed['manual_is_bac']
    overall_acc = reviewed['correct'].mean()
    
    print(f"\n" + "-"*60)
    print(f"Overall Accuracy: {overall_acc:.1%} ({reviewed['correct'].sum()}/{len(reviewed)})")
    print("-"*60)
else:
    print("⚠️  No manual reviews available yet.")
    print("\nTo perform manual review:")
    print("1. Open: data/validation/sample_for_manual_review.csv")
    print("2. For each video, fill 'manual_is_bac' with TRUE or FALSE")
    print("3. Rerun this notebook")

## 4. Visualizations

In [ ]:
# Category distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
category_counts = df['filter_category'].value_counts()
colors = sns.color_palette('husl', len(category_counts))
ax1.pie(category_counts.values, labels=category_counts.index, autopct='%1.1f%%',
        startangle=90, colors=colors)
ax1.set_title('Filter Category Distribution', fontsize=14, fontweight='bold')

# Bar chart
sns.barplot(x=category_counts.index, y=category_counts.values, ax=ax2, palette='husl')
ax2.set_xlabel('Category', fontsize=12)
ax2.set_ylabel('Count', fontsize=12)
ax2.set_title('Videos per Category', fontsize=14, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)

# Add counts on bars
for i, v in enumerate(category_counts.values):
    ax2.text(i, v + 100, f'{v:,}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Confidence distribution by category
fig, ax = plt.subplots(figsize=(12, 6))

sns.violinplot(data=df, x='filter_category', y='filter_confidence', ax=ax, palette='muted')
ax.set_xlabel('Category', fontsize=12)
ax.set_ylabel('Confidence Score', fontsize=12)
ax.set_title('Confidence Score Distribution by Category', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=45)
ax.axhline(y=0.7, color='red', linestyle='--', alpha=0.5, label='Channel Prior Threshold')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Subject distribution (Bac videos only)
fig, ax = plt.subplots(figsize=(12, 6))

subject_counts = bac_videos['subject'].value_counts()
sns.barplot(x=subject_counts.index, y=subject_counts.values, ax=ax, palette='viridis')
ax.set_xlabel('Subject', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Subject Distribution (Bac 3AS Videos Only)', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=45)

# Add percentages on bars
total_bac = len(bac_videos)
for i, v in enumerate(subject_counts.values):
    pct = (v / total_bac) * 100
    ax.text(i, v + 50, f'{v:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Channel performance (top 15 channels by video count)
if 'channel_title' in df.columns:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Top channels by total videos
    top_channels = df['channel_title'].value_counts().head(15)
    sns.barplot(y=top_channels.index, x=top_channels.values, ax=ax1, palette='rocket')
    ax1.set_xlabel('Total Videos', fontsize=12)
    ax1.set_ylabel('Channel', fontsize=12)
    ax1.set_title('Top 15 Channels by Video Count', fontsize=14, fontweight='bold')
    
    # Bac ratio per channel (top 15)
    channel_bac_ratio = df.groupby('channel_title').agg({
        'is_bac_3as': ['sum', 'count']
    })
    channel_bac_ratio.columns = ['bac_count', 'total']
    channel_bac_ratio['bac_ratio'] = channel_bac_ratio['bac_count'] / channel_bac_ratio['total']
    top_bac_channels = channel_bac_ratio.nlargest(15, 'total')
    
    sns.barplot(y=top_bac_channels.index, x=top_bac_channels['bac_ratio'], ax=ax2, palette='coolwarm')
    ax2.set_xlabel('Bac 3AS Ratio', fontsize=12)
    ax2.set_ylabel('Channel', fontsize=12)
    ax2.set_title('Bac 3AS Ratio (Top 15 Channels)', fontsize=14, fontweight='bold')
    ax2.set_xlim(0, 1)
    ax2.axvline(x=0.7, color='red', linestyle='--', alpha=0.5, label='Bac-heavy threshold')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Channel priors visualization
if priors_df is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Bac-heavy vs non-Bac-heavy distribution
    bac_heavy_counts = priors_df['is_bac_heavy'].value_counts()
    colors = ['#2ecc71', '#e74c3c']
    ax1.pie(bac_heavy_counts.values, labels=['Bac-heavy', 'Non-Bac-heavy'], 
            autopct='%1.1f%%', colors=colors, startangle=90)
    ax1.set_title('Channel Classification', fontsize=14, fontweight='bold')
    
    # p_bac distribution
    sns.histplot(priors_df['p_bac'], bins=20, kde=True, ax=ax2, color='steelblue')
    ax2.axvline(x=0.7, color='red', linestyle='--', linewidth=2, label='Bac-heavy threshold (0.7)')
    ax2.set_xlabel('p_bac (Bac ratio)', fontsize=12)
    ax2.set_ylabel('Channel Count', fontsize=12)
    ax2.set_title('Channel Bac Ratio Distribution', fontsize=14, fontweight='bold')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Manual review visualization (if available)
if val_df is not None and val_df['manual_is_bac'].notna().sum() > 0:
    reviewed = val_df.dropna(subset=['manual_is_bac'])
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Precision by category
    metrics_data = []
    for category in ['bac_3as', 'bac_3as_ambiguous', 'non_bac', 'unknown']:
        cat_df = reviewed[reviewed['sample_type'] == category]
        if len(cat_df) > 0:
            if 'bac' in category:
                precision = cat_df['manual_is_bac'].sum() / len(cat_df)
                metrics_data.append({'Category': category, 'Precision': precision, 'n': len(cat_df)})
            else:
                fnr = cat_df['manual_is_bac'].sum() / len(cat_df)
                metrics_data.append({'Category': category, 'FNR': fnr, 'n': len(cat_df)})
    
    metrics_df = pd.DataFrame(metrics_data)
    
    # Precision plot
    if 'Precision' in metrics_df.columns:
        prec_df = metrics_df.dropna(subset=['Precision'])
        bars1 = sns.barplot(x='Category', y='Precision', data=prec_df, ax=ax1, palette='Greens_d')
        ax1.set_ylim(0, 1.1)
        ax1.axhline(y=0.8, color='orange', linestyle='--', label='Target: 80%')
        ax1.axhline(y=0.95, color='green', linestyle='--', label='Target: 95%')
        ax1.set_ylabel('Precision', fontsize=12)
        ax1.set_title('Precision by Category', fontsize=14, fontweight='bold')
        ax1.legend()
        ax1.tick_params(axis='x', rotation=45)
        
        # Add values on bars
        for i, (idx, row) in enumerate(prec_df.iterrows()):
            ax1.text(i, row['Precision'] + 0.03, f"{row['Precision']:.1%}\n(n={row['n']})", 
                    ha='center', va='bottom', fontsize=9)
    
    # Confusion matrix style
    reviewed['correct'] = reviewed['is_bac_3as'] == reviewed['manual_is_bac']
    confusion = pd.crosstab(reviewed['is_bac_3as'], reviewed['manual_is_bac'], 
                            rownames=['Predicted'], colnames=['Actual'])
    sns.heatmap(confusion, annot=True, fmt='d', cmap='Blues', ax=ax2, cbar_kws={'label': 'Count'})
    ax2.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

## 5. Error Analysis

In [ ]:
# Examples of high-confidence Bac videos
print("="*80)
print("HIGH-CONFIDENCE BAC 3AS VIDEOS (Sample)")
print("="*80)

high_conf_bac = df[(df['is_bac_3as'] == True) & (df['filter_confidence'] >= 0.9)].sample(
    min(5, len(df[(df['is_bac_3as'] == True) & (df['filter_confidence'] >= 0.9)])), 
    random_state=42
)

for idx, row in high_conf_bac.iterrows():
    print(f"\nTitle: {row['title'][:100]}...")
    print(f"Category: {row['filter_category']} | Confidence: {row['filter_confidence']:.2f}")
    print(f"Subject: {row['subject']} | Reason: {row['filter_reason'][:60]}...")
    print("-" * 80)

In [ ]:
# Examples of ambiguous cases (medium confidence)
print("="*80)
print("AMBIGUOUS CASES (Confidence 0.6-0.75)")
print("="*80)

ambiguous = df[(df['filter_confidence'] >= 0.6) & (df['filter_confidence'] < 0.75)].sample(
    min(5, len(df[(df['filter_confidence'] >= 0.6) & (df['filter_confidence'] < 0.75)])),
    random_state=42
)

for idx, row in ambiguous.iterrows():
    print(f"\nTitle: {row['title'][:100]}...")
    print(f"Category: {row['filter_category']} | Confidence: {row['filter_confidence']:.2f}")
    print(f"Subject: {row['subject']} | Reason: {row['filter_reason'][:60]}...")
    print("-" * 80)

In [ ]:
# If manual review available: show errors
if val_df is not None and val_df['manual_is_bac'].notna().sum() > 0:
    reviewed = val_df.dropna(subset=['manual_is_bac'])
    reviewed['correct'] = reviewed['is_bac_3as'] == reviewed['manual_is_bac']
    
    errors = reviewed[reviewed['correct'] == False]
    
    if len(errors) > 0:
        print("="*80)
        print(f"CLASSIFICATION ERRORS ({len(errors)} found)")
        print("="*80)
        
        for idx, row in errors.head(10).iterrows():
            print(f"\nTitle: {row['title'][:100]}...")
            print(f"Predicted: {row['is_bac_3as']} | Actual: {row['manual_is_bac']}")
            print(f"Category: {row['filter_category']} | Confidence: {row['filter_confidence']:.2f}")
            print(f"Reason: {row['filter_reason'][:60]}...")
            if pd.notna(row.get('notes')):
                print(f"Notes: {row['notes']}")
            print("-" * 80)
    else:
        print("✓ No errors found in manual review!")

## 6. Export Summary Report

In [ ]:
# Create summary report
summary = {
    'total_videos': len(df),
    'bac_videos': df['is_bac_3as'].sum(),
    'bac_percentage': (df['is_bac_3as'].sum() / len(df)) * 100,
    'categories': df['filter_category'].value_counts().to_dict(),
    'mean_confidence_bac': df[df['is_bac_3as'] == True]['filter_confidence'].mean(),
    'mean_confidence_non_bac': df[df['is_bac_3as'] == False]['filter_confidence'].mean(),
}

if val_df is not None and val_df['manual_is_bac'].notna().sum() > 0:
    reviewed = val_df.dropna(subset=['manual_is_bac'])
    reviewed['correct'] = reviewed['is_bac_3as'] == reviewed['manual_is_bac']
    summary['manual_reviews'] = len(reviewed)
    summary['overall_accuracy'] = reviewed['correct'].mean()

print("="*60)
print("SUMMARY REPORT")
print("="*60)
for key, value in summary.items():
    if isinstance(value, float):
        print(f"{key}: {value:.2f}")
    elif isinstance(value, dict):
        print(f"{key}:")
        for k, v in value.items():
            print(f"  - {k}: {v}")
    else:
        print(f"{key}: {value}")

## Next Steps

**If accuracy is good (>90%):**
- ✓ Use `data/processed/videos_bac_only.csv` for ML modeling
- Continue to `run_pipeline.py engineer`

**If accuracy needs improvement:**
1. Identify error patterns in the analysis above
2. Adjust `config/filter_config.yaml` thresholds
3. Re-run: `uv run python run_pipeline.py filter_data --force-discovery`
4. Re-validate and iterate

## 7. False Negative Deep Dive

**Analysis of videos incorrectly rejected as non-Bac**

Run `scripts/analyze_false_negatives.py` for detailed analysis.

In [ ]:
if val_df is not None and val_df['manual_is_bac'].notna().sum() > 0:
    reviewed = val_df.dropna(subset=['manual_is_bac'])
    
    # False negatives: filter said NOT Bac, but manual says IS Bac
    false_negatives = reviewed[(reviewed['is_bac_3as'] == False) & (reviewed['manual_is_bac'] == True)]
    
    print("=" * 80)
    print(f"FALSE NEGATIVE ANALYSIS: {len(false_negatives)} videos")
    print("=" * 80)
    print("(Videos incorrectly rejected as non-Bac)\n")
    
    if len(false_negatives) > 0:
        # By category
        print("False negatives by category:")
        print(false_negatives['filter_category'].value_counts().to_string())
        
        # Check for existing markers
        print(f"\n{'-'*80}")
        print("Marker presence in false negatives:")
        print("-" * 80)
        
        fn_text = (false_negatives['title'].fillna('') + ' ' + 
                   false_negatives['description'].fillna('')).str.lower()
        
        existing_markers = ['bac', '3as', 'بكالوريا', 'باك', 'ثالثة ثانوي', 'terminale']
        for marker in existing_markers:
            count = fn_text.str.contains(marker, case=False, na=False).sum()
            pct = (count / len(false_negatives)) * 100
            print(f"  '{marker}': {count} videos ({pct:.1f}%)")
        
        # Common terms
        print(f"\n{'-'*80}")
        print("Most common Arabic terms in false negatives:")
        print("-" * 80)
        
        import re
        from collections import Counter
        
        arabic_words = []
        for title in false_negatives['title'].fillna(''):
            words = re.findall(r'[\u0600-\u06FF]+', str(title))
            arabic_words.extend(words)
        
        arabic_counter = Counter(arabic_words)
        for term, count in arabic_counter.most_common(15):
            pct = (count / len(false_negatives)) * 100
            print(f"  {term:20s} : {count:3d} ({pct:.1f}%)")
        
        # Sample titles
        print(f"\n{'-'*80}")
        print("Sample false negative titles (first 10):")
        print("-" * 80 + "\n")
        
        for i, (idx, row) in enumerate(false_negatives.head(10).iterrows(), 1):
            print(f"{i:2d}. {row['title'][:80]}")
            print(f"    Category: {row['filter_category']} | Confidence: {row['filter_confidence']:.2f}")
            print(f"    Reason: {row['filter_reason'][:60]}")
            print()
    else:
        print("✓ No false negatives found!")
else:
    print("⚠️ Manual review required for false negative analysis")

## 8. Channel-Level Analysis

**Identify problematic channels based on manual review accuracy**

This analysis helps detect:
- Channels with many false positives (broad/non-specialized content)
- Channels with inconsistent Bac content
- Channels that should be removed to reduce noise

In [ ]:
# Fixed version of channel analysis cell
# Copy this code to replace Cell 53 in your notebook

if val_df is not None and val_df['manual_is_bac'].notna().sum() > 0:
    reviewed = val_df.dropna(subset=['manual_is_bac'])
    
    print("=" * 80)
    print("CHANNEL-LEVEL ACCURACY ANALYSIS")
    print("=" * 80)
    print("\nThis analysis identifies channels with low accuracy (potential noise sources)\n")
    
    # Calculate metrics per channel using a cleaner approach
    channel_results = []
    
    for channel_name in reviewed['channel_title'].unique():
        channel_data = reviewed[reviewed['channel_title'] == channel_name].copy()
        
        total = len(channel_data)
        actual_bac = channel_data['manual_is_bac'].sum()
        predicted_bac = channel_data['is_bac_3as'].sum()
        
        # Calculate accuracy
        correct = (channel_data['is_bac_3as'] == channel_data['manual_is_bac']).sum()
        accuracy = correct / total if total > 0 else 0
        
        # False positive rate: of videos predicted as Bac, how many are actually not?
        if predicted_bac > 0:
            false_positives = ((channel_data['is_bac_3as'] == True) & (channel_data['manual_is_bac'] == False)).sum()
            fp_rate = false_positives / predicted_bac
        else:
            fp_rate = 0
        
        # False negative rate: of videos predicted as non-Bac, how many are actually Bac?
        predicted_non_bac = total - predicted_bac
        if predicted_non_bac > 0:
            false_negatives = ((channel_data['is_bac_3as'] == False) & (channel_data['manual_is_bac'] == True)).sum()
            fn_rate = false_negatives / predicted_non_bac
        else:
            fn_rate = 0
        
        channel_results.append({
            'channel_title': channel_name,
            'total_reviewed': total,
            'actual_bac': actual_bac,
            'predicted_bac': predicted_bac,
            'accuracy': accuracy,
            'false_positive_rate': fp_rate,
            'false_negative_rate': fn_rate
        })
    
    # Create DataFrame
    channel_stats = pd.DataFrame(channel_results)
    
    # Sort by accuracy (worst first)
    channel_stats_sorted = channel_stats.sort_values('accuracy')
    
    print(f"\nTotal channels in validation: {len(channel_stats)}")
    print(f"Average accuracy: {channel_stats['accuracy'].mean():.1%}")
    print(f"\n{'-'*80}")
    print("WORST PERFORMING CHANNELS (Lowest Accuracy)")
    print("-" * 80)
    print(channel_stats_sorted[['channel_title', 'total_reviewed', 'accuracy', 'false_positive_rate']].head(10).to_string(index=False))
    
    print(f"\n{'-'*80}")
    print("BEST PERFORMING CHANNELS (Highest Accuracy)")
    print("-" * 80)
    print(channel_stats_sorted[['channel_title', 'total_reviewed', 'accuracy', 'false_positive_rate']].tail(10).to_string(index=False))
    
    # Identify problematic channels
    problematic_threshold = 0.7  # <70% accuracy
    problematic = channel_stats[channel_stats['accuracy'] < problematic_threshold]
    
    if len(problematic) > 0:
        print(f"\n{'-'*80}")
        print(f"⚠️  PROBLEMATIC CHANNELS (Accuracy < {problematic_threshold:.0%})")
        print("-" * 80)
        print(f"\n{len(problematic)} channels have accuracy below {problematic_threshold:.0%}:\n")
        
        for idx, row in problematic.iterrows():
            print(f"  • {row['channel_title']}")
            print(f"    - Accuracy: {row['accuracy']:.1%} ({row['total_reviewed']} samples)")
            print(f"    - False Positive Rate: {row['false_positive_rate']:.1%}")
            print(f"    - Predicted Bac: {row['predicted_bac']}, Actual Bac: {row['actual_bac']}")
            print()
        
        print(f"\n{'='*80}")
        print("RECOMMENDATION")
        print("=" * 80)
        print("\nConsider removing these channels from data/raw/channels.csv:")
        print("They are producing significant noise in your Bac dataset.\n")
        
        for channel in problematic['channel_title'].values:
            print(f"  - {channel}")
    else:
        print(f"\n✓ All channels have accuracy ≥ {problematic_threshold:.0%}")
else:
    print("⚠️ Manual review required for channel-level analysis")


In [ ]:
# Channel performance visualizations
if val_df is not None and val_df['manual_is_bac'].notna().sum() > 0:
    reviewed = val_df.dropna(subset=['manual_is_bac'])
    # Calculate metrics per channel (FIXED - no groupby alignment issues)
    channel_results = []

    for channel_name in reviewed['channel_title'].unique():
        channel_data = reviewed[reviewed['channel_title'] == channel_name].copy()
        
        total = len(channel_data)
        actual_bac = channel_data['manual_is_bac'].sum()
        predicted_bac = channel_data['is_bac_3as'].sum()
        
        # Calculate accuracy
        correct = (channel_data['is_bac_3as'] == channel_data['manual_is_bac']).sum()
        accuracy = correct / total if total > 0 else 0
        
        channel_results.append({
            'channel_title': channel_name,
            'total_reviewed': total,
            'actual_bac': actual_bac,
            'predicted_bac': predicted_bac,
            'accuracy': accuracy,
        })

    # Create DataFrame
    channel_stats = pd.DataFrame(channel_results)
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Accuracy distribution across channels
    ax1 = axes[0, 0]
    channel_stats_sorted = channel_stats.sort_values('accuracy')
    colors = ['red' if acc < 0.7 else 'orange' if acc < 0.8 else 'green' for acc in channel_stats_sorted['accuracy']]
    
    ax1.barh(range(len(channel_stats_sorted)), channel_stats_sorted['accuracy'], color=colors, alpha=0.7)
    ax1.set_yticks(range(len(channel_stats_sorted)))
    ax1.set_yticklabels(channel_stats_sorted['channel_title'], fontsize=8)
    ax1.set_xlabel('Accuracy', fontsize=12)
    ax1.set_title('Channel Accuracy (Worst to Best)', fontsize=14, fontweight='bold')
    ax1.axvline(x=0.7, color='red', linestyle='--', alpha=0.5, label='70% threshold')
    ax1.axvline(x=0.8, color='orange', linestyle='--', alpha=0.5, label='80% threshold')
    ax1.legend()
    ax1.set_xlim(0, 1)
    ax1.grid(axis='x', alpha=0.3)
    
    # 2. Accuracy distribution histogram
    ax2 = axes[0, 1]
    ax2.hist(channel_stats['accuracy'], bins=10, color='steelblue', edgecolor='black', alpha=0.7)
    ax2.axvline(x=channel_stats['accuracy'].mean(), color='red', linestyle='--', 
                linewidth=2, label=f'Mean: {channel_stats["accuracy"].mean():.1%}')
    ax2.axvline(x=0.7, color='orange', linestyle='--', linewidth=2, label='70% threshold')
    ax2.set_xlabel('Accuracy', fontsize=12)
    ax2.set_ylabel('Number of Channels', fontsize=12)
    ax2.set_title('Channel Accuracy Distribution', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    # 3. Samples per channel
    ax3 = axes[1, 0]
    channel_samples = channel_stats.sort_values('total_reviewed', ascending=False).head(15)
    ax3.bar(range(len(channel_samples)), channel_samples['total_reviewed'], 
            color='teal', alpha=0.7)
    ax3.set_xticks(range(len(channel_samples)))
    ax3.set_xticklabels(channel_samples['channel_title'], rotation=45, ha='right', fontsize=8)
    ax3.set_ylabel('Number of Samples', fontsize=12)
    ax3.set_title('Top 15 Channels by Sample Count', fontsize=14, fontweight='bold')
    ax3.grid(axis='y', alpha=0.3)
    
    # 4. Accuracy vs Sample Size
    ax4 = axes[1, 1]
    scatter = ax4.scatter(channel_stats['total_reviewed'], channel_stats['accuracy'], 
                         s=100, alpha=0.6, c=channel_stats['accuracy'], 
                         cmap='RdYlGn', edgecolors='black', linewidth=0.5)
    ax4.axhline(y=0.7, color='red', linestyle='--', alpha=0.5, label='70% threshold')
    ax4.set_xlabel('Number of Samples Reviewed', fontsize=12)
    ax4.set_ylabel('Accuracy', fontsize=12)
    ax4.set_title('Channel Accuracy vs Sample Size', fontsize=14, fontweight='bold')
    ax4.legend()
    ax4.grid(alpha=0.3)
    
    # Add colorbar
    plt.colorbar(scatter, ax=ax4, label='Accuracy')
    
    # Annotate problematic channels (low accuracy, high sample count)
    problematic = channel_stats[(channel_stats['accuracy'] < 0.7) & (channel_stats['total_reviewed'] >= 5)]
    for idx, row in problematic.iterrows():
        ax4.annotate(row['channel_title'][:20], 
                    xy=(row['total_reviewed'], row['accuracy']),
                    xytext=(5, 5), textcoords='offset points',
                    fontsize=7, alpha=0.8, 
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.3))
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Manual review required for visualizations")

In [ ]:
# Channel performance visualizations - FIXED VERSION
if val_df is not None and val_df['manual_is_bac'].notna().sum() > 0:
    reviewed = val_df.dropna(subset=['manual_is_bac'])
    
    # Calculate metrics per channel (FIXED - no groupby alignment issues)
    channel_results = []
    
    for channel_name in reviewed['channel_title'].unique():
        channel_data = reviewed[reviewed['channel_title'] == channel_name].copy()
        
        total = len(channel_data)
        actual_bac = channel_data['manual_is_bac'].sum()
        predicted_bac = channel_data['is_bac_3as'].sum()
        
        # Calculate accuracy
        correct = (channel_data['is_bac_3as'] == channel_data['manual_is_bac']).sum()
        accuracy = correct / total if total > 0 else 0
        
        channel_results.append({
            'channel_title': channel_name,
            'total_reviewed': total,
            'actual_bac': actual_bac,
            'predicted_bac': predicted_bac,
            'accuracy': accuracy,
        })
    
    # Create DataFrame
    channel_stats = pd.DataFrame(channel_results)
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Accuracy distribution across channels
    ax1 = axes[0, 0]
    channel_stats_sorted = channel_stats.sort_values('accuracy')
    colors = ['red' if acc < 0.7 else 'orange' if acc < 0.8 else 'green' for acc in channel_stats_sorted['accuracy']]
    
    ax1.barh(range(len(channel_stats_sorted)), channel_stats_sorted['accuracy'], color=colors, alpha=0.7)
    ax1.set_yticks(range(len(channel_stats_sorted)))
    ax1.set_yticklabels(channel_stats_sorted['channel_title'], fontsize=8)
    ax1.set_xlabel('Accuracy', fontsize=12)
    ax1.set_title('Channel Accuracy (Worst to Best)', fontsize=14, fontweight='bold')
    ax1.axvline(x=0.7, color='red', linestyle='--', alpha=0.5, label='70% threshold')
    ax1.axvline(x=0.8, color='orange', linestyle='--', alpha=0.5, label='80% threshold')
    ax1.legend()
    ax1.set_xlim(0, 1)
    ax1.grid(axis='x', alpha=0.3)
    
    # 2. Accuracy distribution histogram
    ax2 = axes[0, 1]
    ax2.hist(channel_stats['accuracy'], bins=10, color='steelblue', edgecolor='black', alpha=0.7)
    ax2.axvline(x=channel_stats['accuracy'].mean(), color='red', linestyle='--', 
                linewidth=2, label=f'Mean: {channel_stats["accuracy"].mean():.1%}')
    ax2.axvline(x=0.7, color='orange', linestyle='--', linewidth=2, label='70% threshold')
    ax2.set_xlabel('Accuracy', fontsize=12)
    ax2.set_ylabel('Number of Channels', fontsize=12)
    ax2.set_title('Channel Accuracy Distribution', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    # 3. Samples per channel
    ax3 = axes[1, 0]
    channel_samples = channel_stats.sort_values('total_reviewed', ascending=False).head(15)
    ax3.bar(range(len(channel_samples)), channel_samples['total_reviewed'], 
            color='teal', alpha=0.7)
    ax3.set_xticks(range(len(channel_samples)))
    ax3.set_xticklabels(channel_samples['channel_title'], rotation=45, ha='right', fontsize=8)
    ax3.set_ylabel('Number of Samples', fontsize=12)
    ax3.set_title('Top 15 Channels by Sample Count', fontsize=14, fontweight='bold')
    ax3.grid(axis='y', alpha=0.3)
    
    # 4. Accuracy vs Sample Size
    ax4 = axes[1, 1]
    scatter = ax4.scatter(channel_stats['total_reviewed'], channel_stats['accuracy'], 
                         s=100, alpha=0.6, c=channel_stats['accuracy'], 
                         cmap='RdYlGn', edgecolors='black', linewidth=0.5)
    ax4.axhline(y=0.7, color='red', linestyle='--', alpha=0.5, label='70% threshold')
    ax4.set_xlabel('Number of Samples Reviewed', fontsize=12)
    ax4.set_ylabel('Accuracy', fontsize=12)
    ax4.set_title('Channel Accuracy vs Sample Size', fontsize=14, fontweight='bold')
    ax4.legend()
    ax4.grid(alpha=0.3)
    
    # Add colorbar
    plt.colorbar(scatter, ax=ax4, label='Accuracy')
    
    # Annotate problematic channels (low accuracy, high sample count)
    problematic = channel_stats[(channel_stats['accuracy'] < 0.7) & (channel_stats['total_reviewed'] >= 5)]
    for idx, row in problematic.iterrows():
        ax4.annotate(row['channel_title'][:20], 
                    xy=(row['total_reviewed'], row['accuracy']),
                    xytext=(5, 5), textcoords='offset points',
                    fontsize=7, alpha=0.8, 
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.3))
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Manual review required for visualizations")

In [ ]:
# Export channel analysis for decision making - FIXED VERSION
if val_df is not None and val_df['manual_is_bac'].notna().sum() > 0:
    reviewed = val_df.dropna(subset=['manual_is_bac'])
    
    # Calculate metrics per channel - FIXED (no groupby alignment issues)
    channel_results = []
    
    for channel_name in reviewed['channel_title'].unique():
        channel_data = reviewed[reviewed['channel_title'] == channel_name].copy()
        
        # Get channel_id (first occurrence)
        channel_id = channel_data['channel_id'].iloc[0]
        
        total = len(channel_data)
        actual_bac = channel_data['manual_is_bac'].sum()
        predicted_bac = channel_data['is_bac_3as'].sum()
        
        # Calculate accuracy
        correct = (channel_data['is_bac_3as'] == channel_data['manual_is_bac']).sum()
        accuracy = correct / total if total > 0 else 0
        
        # Calculate percentages
        actual_bac_pct = (actual_bac / total) * 100 if total > 0 else 0
        predicted_bac_pct = (predicted_bac / total) * 100 if total > 0 else 0
        
        # Add recommendation
        if accuracy < 0.6:
            recommendation = "⚠️ REMOVE - Very poor accuracy"
        elif accuracy < 0.7:
            recommendation = "⚠️ REVIEW - Low accuracy"
        elif accuracy < 0.8:
            recommendation = "⚠️ MONITOR - Moderate accuracy"
        else:
            recommendation = "✓ KEEP - Good accuracy"
        
        channel_results.append({
            'channel_title': channel_name,
            'channel_id': channel_id,
            'total_reviewed': total,
            'actual_bac': actual_bac,
            'predicted_bac': predicted_bac,
            'accuracy': accuracy,
            'actual_bac_pct': actual_bac_pct,
            'predicted_bac_pct': predicted_bac_pct,
            'recommendation': recommendation
        })
    
    # Create DataFrame
    channel_stats = pd.DataFrame(channel_results)
    
    # Sort by accuracy
    channel_stats_sorted = channel_stats.sort_values('accuracy')
    
    # Save to CSV for review
    export_path = DATA_DIR / 'validation' / 'channel_accuracy_report.csv'
    channel_stats_sorted.to_csv(export_path, index=False)
    
    print("=" * 80)
    print("CHANNEL ACCURACY REPORT EXPORTED")
    print("=" * 80)
    print(f"\nSaved to: {export_path}")
    print(f"\nSummary:")
    print(f"  • Total channels: {len(channel_stats)}")
    print(f"  • Channels to REMOVE (<60%): {(channel_stats['accuracy'] < 0.6).sum()}")
    print(f"  • Channels to REVIEW (60-70%): {((channel_stats['accuracy'] >= 0.6) & (channel_stats['accuracy'] < 0.7)).sum()}")
    print(f"  • Channels to MONITOR (70-80%): {((channel_stats['accuracy'] >= 0.7) & (channel_stats['accuracy'] < 0.8)).sum()}")
    print(f"  • Channels to KEEP (≥80%): {(channel_stats['accuracy'] >= 0.8).sum()}")
    
    print("\n" + "-" * 80)
    print("Top recommendations for channel cleanup:")
    print("-" * 80)
    
    remove_channels = channel_stats_sorted[channel_stats_sorted['accuracy'] < 0.6]
    if len(remove_channels) > 0:
        print(f"\n🗑️  Remove {len(remove_channels)} channels with <60% accuracy:\n")
        for idx, row in remove_channels.iterrows():
            print(f"  • {row['channel_title']:<40} (Accuracy: {row['accuracy']:.1%}, Samples: {row['total_reviewed']})")
            print(f"    Channel ID: {row['channel_id']}")
    
    review_channels = channel_stats_sorted[(channel_stats_sorted['accuracy'] >= 0.6) & (channel_stats_sorted['accuracy'] < 0.7)]
    if len(review_channels) > 0:
        print(f"\n⚠️  Review {len(review_channels)} channels with 60-70% accuracy:\n")
        for idx, row in review_channels.iterrows():
            print(f"  • {row['channel_title']:<40} (Accuracy: {row['accuracy']:.1%}, Samples: {row['total_reviewed']})")
    
    print(f"\n\nOpen {export_path.name} to see full details and make decisions.")
else:
    print("⚠️ Manual review required to generate channel accuracy report")

In [ ]:
# In your notebook, add this analysis:
for channel in ['الأستاذ شريفي عربية', 'الأستاذ عبد الباسط', 
                'الاستاذ مرنيز وليد للرياضيات', 'أيوب للعلوم', 
                'الأستاذ شاوش', 'الأستاذ عبدالنور خليفي Abdennour Khalifi']:
    
    channel_errors = reviewed[(reviewed['channel_title'] == channel) & 
                              (reviewed['is_bac_3as'] != reviewed['manual_is_bac'])]
    
    print(f"\n{channel}: {len(channel_errors)} errors")
    for idx, row in channel_errors.head(3).iterrows():
        print(f"  - {row['title'][:60]}")
        print(f"    Reason: {row['filter_reason'][:50]}")